# F1 Tyre-Aware Pit Stop Engine: training data and first results

This notebook downloads real Formula 1 race data with **FastF1** (and, for races FastF1 cannot deliver on
Colab, the same lap tables from the TracingInsights archive on GitHub), cleans and labels it, shows what the
training data looks like, and trains a **first baseline model** (XGBoost) to predict pit-stop laps.

**How to run:** `Runtime → Run all`. The first run uses `QUICK = True` (4 races per season) so you can check
everything works in a few minutes. Then set `QUICK = False` and run again for the full results.

**Data split used here:**
- Training: 2022 and 2023 seasons
- Validation: 2024 season
- 2025 and 2026 are **not touched** here; they are kept for the final test, as the proposal says.

The model in this notebook is the **baseline**, not the final two-stage LSTM. Its scores are the bar the final model must beat.

## 1. Install

In [ ]:
!pip install -q fastf1 xgboost pyarrow pandas

## 2. Settings

Saving to Google Drive means the download only happens once: later runs read your saved files.

In [ ]:
import os, json, warnings
warnings.filterwarnings("ignore")

USE_DRIVE = True          # False = save inside this Colab session only (lost when it closes)
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/f1_project"
else:
    BASE = "/content/f1_project"

CACHE_DIR = f"{BASE}/fastf1_cache"   # FastF1's own download cache
DATA_DIR  = f"{BASE}/races"          # one cleaned Parquet file per race
OUT_DIR   = f"{BASE}/results"        # charts, tables and metrics
for d in (CACHE_DIR, DATA_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)

TRAIN_SEASONS = [2022, 2023]   # the model learns from these
VALID_SEASON  = 2024           # the model is checked on this
QUICK = True                   # True = first QUICK_RACES races per season, for a fast test run
QUICK_RACES = 4
RANDOM_SEED = 42

# Races where pit stops were forced by a special rule, not tyre wear (see proposal)
RULE_FORCED = {(2023, "Qatar"), (2025, "Monaco"), (2025, "Qatar")}

2.5**  Checking File Status **

In [ ]:
import os

print(f"Checking directory: {DATA_DIR}")
if os.path.exists(DATA_DIR):
    files = sorted(os.listdir(DATA_DIR))
    print(f"Total files found: {len(files)}")
    for f in files:
        print("  -", f)
else:
    print("Directory does not exist yet.")

### 2.6 Check that every saved race is a real race

A file such as `2023_01.parquet` must hold the laps of the real 2023 round 1. A 2022 race saved under a
2023 or 2024 name is not data from that season: the model would be tested on the races it was trained on,
and every result below would be meaningless. This cell compares the saved races with each other and stops
if two of them have the same drivers, tyres and positions on every lap.

In [ ]:
import hashlib
import numpy as np
import pandas as pd

DELETE_COPIES = False     # set to True and run this cell again to delete the copied files listed below

FINGERPRINT_COLS = ["Driver", "LapNumber", "Stint", "Compound", "TyreLife", "Position"]

def race_fingerprint(race):
    """The same value for two races only when every driver has the same tyres and position on every lap."""
    cols = [c for c in FINGERPRINT_COLS if c in race.columns]
    part = race[cols].sort_values(["Driver", "LapNumber"]).reset_index(drop=True)
    return hashlib.sha1(pd.util.hash_pandas_object(part, index=False).to_numpy().tobytes()).hexdigest()

def copied_races(races):
    """races: {name: laps}. Returns [(copy, original)]; the earliest name of each identical group is kept."""
    seen, copies = {}, []
    for name in sorted(races):
        if len(races[name]) == 0:
            continue
        key = race_fingerprint(races[name])
        if key in seen:
            copies.append((name, seen[key]))
        else:
            seen[key] = name
    return copies

saved = {f[:-8]: pd.read_parquet(f"{DATA_DIR}/{f}") for f in sorted(os.listdir(DATA_DIR)) if f.endswith(".parquet")}
copies = copied_races(saved)
print(f"{len(saved)} saved race file(s) in {DATA_DIR}")
for copy, original in copies:
    print(f"  COPY  {copy}.parquet has the same laps as {original}.parquet")
if copies and DELETE_COPIES:
    for copy, _ in copies:
        os.remove(f"{DATA_DIR}/{copy}.parquet")
        print(f"  deleted {copy}.parquet")
elif copies:
    raise RuntimeError(
        f"{len(copies)} saved race(s) are copies of another race, not real data. "
        "Set DELETE_COPIES = True in this cell and run it again to delete them, then continue."
    )
else:
    print("No copied races found.")

## 3. Download race data

Each race is downloaded once and saved. If a race fails (for example a server timeout), it is skipped
and listed at the end; just run this cell again later to fill the gaps.

In [ ]:
import fastf1
import numpy as np
import pandas as pd
import os
import logging

fastf1.Cache.enable_cache(CACHE_DIR)
fastf1.set_log_level("ERROR")
logging.getLogger("fastf1").propagate = False   # stops Colab printing FastF1's debug lines a second time

BLOCKED_HELP = (
    "Several races in a row could not be loaded from FastF1 or from the GitHub archive. Check the internet "
    "connection of this session and run the cell again. If it still fails, download the races on your own "
    "computer (see 'If the download is blocked' below this cell) and copy the .parquet files into "
    + DATA_DIR + ". Saved races are read from disk and are not downloaded again."
)
MAX_FAILS_IN_A_ROW = 3

KEEP_COLS = ["Driver", "Team", "LapNumber", "Stint", "Compound", "TyreLife", "FreshTyre",
             "TrackStatus", "Position", "IsAccurate", "Deleted", "FastF1Generated"]
TIME_COLS = ["LapTime", "Time", "PitInTime", "PitOutTime"]
WEATHER_COLS = ["AirTemp", "TrackTemp", "Humidity", "Rainfall"]

import json as _json, urllib.request, urllib.parse, urllib.error
from concurrent.futures import ThreadPoolExecutor

# Second source: the TracingInsights archive on GitHub (MIT licence). It holds exports of FastF1's own lap
# table, so it is the same real timing data, and Colab can reach it. Used only when FastF1 returns nothing.
ARCHIVE_URL = "https://raw.githubusercontent.com/TracingInsights-Archive/{year}/HEAD/{event}/Race/{name}"
ARCHIVE_NUMBERS = {"lap": "LapNumber", "stint": "Stint", "life": "TyreLife", "pos": "Position",
                   "time": "LapTime_s", "sesT": "Time_s", "pin": "PitInTime_s", "pout": "PitOutTime_s",
                   "wAT": "AirTemp", "wTT": "TrackTemp", "wH": "Humidity"}
ARCHIVE_BOOLS = {"fresh": "FreshTyre", "iacc": "IsAccurate", "del": "Deleted", "ff1G": "FastF1Generated",
                 "wR": "Rainfall"}

def archive_json(year, event, name):
    url = ARCHIVE_URL.format(year=year, event=urllib.parse.quote(event), name=urllib.parse.quote(name))
    with urllib.request.urlopen(url, timeout=30) as response:
        return _json.loads(response.read().decode("utf-8"))

def load_race_from_archive(year, rnd, event):
    """One race from the archive, in the same columns as a FastF1 download."""
    drivers = [d["driver"] for d in archive_json(year, event, "drivers.json")["drivers"]]
    def one(driver):
        try:
            return driver, archive_json(year, event, f"{driver}/laptimes.json")
        except urllib.error.HTTPError as e:
            if e.code == 404:          # listed but did not start: no lap file
                return driver, None
            raise
    with ThreadPoolExecutor(max_workers=8) as pool:
        files = list(pool.map(one, drivers))
    parts = []
    for driver, data in files:
        if not data or not len(data.get("lap", [])):
            continue
        clean = lambda key: pd.Series(data[key], dtype=object).where(lambda s: ~s.isin(["None", "nan", "NaN", ""]), np.nan)
        part = pd.DataFrame({"Driver": [driver] * len(data["lap"])})
        part["Team"] = clean("team").to_numpy() if "team" in data else np.nan
        for key, col in ARCHIVE_NUMBERS.items():
            part[col] = pd.to_numeric(clean(key), errors="coerce").to_numpy(float) if key in data else np.nan
        for key, col in ARCHIVE_BOOLS.items():
            part[col] = clean(key).map(lambda v: v is True or v == "True").astype(bool).to_numpy() if key in data else False
        part["Compound"] = clean("compound").astype(str).str.upper().to_numpy()
        part["TrackStatus"] = clean("status").astype(str).to_numpy()
        parts.append(part)
    if not parts:
        raise ValueError("the archive has no lap files for this race")
    df = pd.concat(parts, ignore_index=True).dropna(subset=["LapNumber"])
    df["Year"], df["Round"], df["EventName"] = year, rnd, event
    df["RaceID"], df["Source"] = f"{year}_{rnd:02d}", "tracinginsights"
    return df.sort_values(["Driver", "LapNumber"]).reset_index(drop=True)

USE_FASTF1 = True     # switched off automatically after 3 failures in a row, to save time
FASTF1_FAILS = 0

def load_race(year, rnd, event=None):
    """Read the saved copy, else download with FastF1, else read the race from the GitHub archive."""
    global USE_FASTF1, FASTF1_FAILS
    path = f"{DATA_DIR}/{year}_{rnd:02d}.parquet"
    if os.path.exists(path):
        return pd.read_parquet(path)
    first_error = "skipped"
    if USE_FASTF1:
        try:
            df = load_race_with_fastf1(year, rnd)
            FASTF1_FAILS = 0
            df.to_parquet(path, index=False)
            return df
        except Exception as e:
            first_error = f"{type(e).__name__}: {str(e)[:60]}"
            FASTF1_FAILS += 1
            if FASTF1_FAILS >= MAX_FAILS_IN_A_ROW:
                USE_FASTF1 = False
                print("      FastF1 returned no data 3 times in a row; the remaining races are read from the archive.")
    if not event:
        raise ValueError(f"FastF1: {first_error}; no event name for the archive")
    try:
        df = load_race_from_archive(year, rnd, event)
    except Exception as e:
        raise ValueError(f"FastF1: {first_error}; archive: {type(e).__name__}: {str(e)[:60]}")
    df.to_parquet(path, index=False)
    return df

def load_race_with_fastf1(year, rnd):
    """Download one race with FastF1 and return one row per lap."""
    session = fastf1.get_session(year, rnd, "R")
    session.load(laps=True, telemetry=False, weather=True, messages=True)
    try:
        laps = session.laps
    except Exception:
        laps = None
    if laps is None or len(laps) == 0:
        raise ValueError("no lap data returned")

    weather = laps.get_weather_data().reset_index(drop=True)
    laps = pd.DataFrame(laps).reset_index(drop=True)

    df = pd.DataFrame({c: laps[c] for c in KEEP_COLS if c in laps.columns})
    for c in TIME_COLS:                                   # timedeltas -> seconds
        df[c + "_s"] = pd.to_timedelta(laps[c]).dt.total_seconds()
    for c in WEATHER_COLS:
        df[c] = weather[c].values if c in weather.columns else np.nan
    for c in ["IsAccurate", "Deleted", "FastF1Generated", "FreshTyre", "Rainfall"]:
        if c in df.columns:
            df[c] = df[c].fillna(False).astype(bool)
    df["Compound"] = df["Compound"].astype(str).str.upper()
    df["TrackStatus"] = df["TrackStatus"].astype(str)
    df["Year"], df["Round"] = year, rnd
    df["EventName"] = str(session.event["EventName"])
    df["RaceID"] = f"{year}_{rnd:02d}"
    df["Source"] = "fastf1"
    return df

frames, failed, stop_reason, fails_in_a_row = [], [], None, 0
for year in TRAIN_SEASONS + [VALID_SEASON]:
    schedule = fastf1.get_event_schedule(year, include_testing=False)
    rounds = [int(r) for r in schedule["RoundNumber"] if int(r) > 0]
    if "EventDate" in schedule.columns:
        done = pd.to_datetime(schedule["EventDate"]) < pd.Timestamp.now()
        rounds = [int(r) for r in schedule.loc[done, "RoundNumber"] if int(r) > 0]
    if QUICK:
        rounds = rounds[:QUICK_RACES]
    names = ({int(r): str(n) for r, n in zip(schedule["RoundNumber"], schedule["EventName"])}
             if "EventName" in schedule.columns else {})
    for rnd in rounds:
        try:
            frames.append(load_race(year, rnd, names.get(rnd)))
            fails_in_a_row = 0
            src = frames[-1]["Source"].iloc[0] if "Source" in frames[-1].columns else "fastf1"
            print(f"OK    {year} round {rnd:2d}  ({src})")
        except Exception as e:
            fails_in_a_row += 1
            failed.append((year, rnd, f"{type(e).__name__}: {str(e)[:80]}"))
            print(f"SKIP  {year} round {rnd:2d}: {type(e).__name__}: {str(e)[:80]}")
            if "ratelimit" in type(e).__name__.lower() or "rate limit" in str(e).lower():
                stop_reason = ("The F1 data service limits how many downloads it allows per hour and the limit "
                               "was reached. Wait about an hour, then run this cell again.")
            elif fails_in_a_row >= MAX_FAILS_IN_A_ROW:
                stop_reason = BLOCKED_HELP
            if stop_reason:
                break
    if stop_reason:
        break

if frames:
    raw = pd.concat(frames, ignore_index=True)
    raw["RaceID"] = raw["Year"].astype(int).astype(str) + "_" + raw["Round"].astype(int).astype(str).str.zfill(2)
    if "Source" not in raw.columns:
        raw["Source"] = "fastf1"
    raw["Source"] = raw["Source"].fillna("fastf1")          # races saved before the source was recorded
    print("Races by source:", raw.drop_duplicates("RaceID")["Source"].value_counts().to_dict())
    per_season = raw.groupby("Year")["Round"].nunique()
    print(f"\nLoaded {raw[['Year','Round']].drop_duplicates().shape[0]} races, {len(raw):,} laps. Failed: {len(failed)}")
else:
    per_season = pd.Series(dtype=int)
print("Races per season:", {int(y): int(per_season.get(y, 0)) for y in TRAIN_SEASONS + [VALID_SEASON]})

missing = [y for y in TRAIN_SEASONS + [VALID_SEASON] if per_season.get(y, 0) == 0]
if not frames:
    raise RuntimeError("No race could be loaded. " + (stop_reason or "Check the SKIP lines above."))
if stop_reason and "limit" in stop_reason and "hour" in stop_reason:
    raise RuntimeError(stop_reason + " Races already saved are reused, so only the missing ones are downloaded.")

# SPLIT_MODE decides how training and validation races are chosen further down:
#   "by_season"  the planned split: train on TRAIN_SEASONS, validate on VALID_SEASON
#   "by_race"    used only when a whole season is missing: the LAST races of the real data that did load
#                are held out for validation. Every lap is still real; nothing is copied or invented.
SPLIT_MODE = "by_race" if missing else "by_season"
if missing:
    print(f"\nWARNING: no races could be loaded for {missing}. {stop_reason or ''}")
    print("The notebook continues with the real races that did load and holds out the LAST races for")
    print("validation (SPLIT_MODE = 'by_race'). These are practice results on a small dataset, not the")
    print("project's planned 2022-2023 -> 2024 validation. Missing seasons are never filled with copies.")
elif stop_reason:
    print(f"\nWARNING: some races could not be downloaded ({len(failed)} failed). Continuing with the rest.")

copies = copied_races({rid: race for rid, race in raw.groupby("RaceID")})
if copies:
    raise RuntimeError("These races have the same laps as another race, so they are copies, not real data: "
                       + ", ".join(f"{c} (= {o})" for c, o in copies)
                       + ". Run the check cell above with DELETE_COPIES = True, then run this cell again.")


def split_labels(d, valid_share=0.25):
    """'train' / 'validation' / 'other' for every lap. Call it on the CLEANED laps (dry races only)."""
    global VALID_RACES, VALID_LABEL, VALID_TITLE
    if SPLIT_MODE == "by_season":
        labels = np.where(d["Year"].isin(TRAIN_SEASONS), "train",
                          np.where(d["Year"] == VALID_SEASON, "validation", "other"))
        VALID_RACES = sorted(d.loc[labels == "validation", "RaceID"].unique())
        VALID_LABEL, VALID_TITLE = f"season {VALID_SEASON}", f"the {VALID_SEASON} season"
        return labels
    usable = d if "RuleForced" not in d.columns else d[~d["RuleForced"]]
    races = sorted(usable["RaceID"].unique())          # "2022_01" < "2022_02" < ... : race order
    if len(races) < 3:
        raise RuntimeError(
            f"Only {len(races)} dry race(s) are available. At least 3 are needed (2 to train, 1 to check). "
            "Set QUICK = False to try more races, or download the races on your own computer "
            "(see 'If the download is blocked')."
        )
    n_valid = max(1, int(round(valid_share * len(races))))
    VALID_RACES = races[-n_valid:]
    VALID_LABEL = f"races {', '.join(VALID_RACES)} (held out; practice split, a season is missing)"
    VALID_TITLE = f"{n_valid} held-out race(s)"
    return np.where(d["RaceID"].isin(VALID_RACES), "validation", "train")


In [ ]:
# Calculate missing values in the raw DataFrame
missing_count = raw.isnull().sum()
missing_percent = (raw.isnull().sum() / len(raw)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing_count,
    'Percentage (%)': missing_percent
}).sort_values(by='Missing Values', ascending=False)

display(missing_df)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the visualization style
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

# Filter out extreme outliers (e.g., lap times > 150s, which are typically slow pit laps or red flags)
# This helps focus the histogram on the active racing lap times.
active_laps = raw[raw['LapTime_s'].between(60, 120)]

sns.histplot(data=active_laps, x='LapTime_s', bins=50, kde=True, color='#08519c')

plt.title('Distribution of F1 Lap Times (Active Racing Laps: 60s - 120s)', fontsize=14)
plt.xlabel('Lap Time (seconds)', fontsize=12)
plt.ylabel('Lap Count', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Identify top drivers by lap count to keep the boxplot readable
top_drivers = raw['Driver'].value_counts().head(12).index
driver_laps = raw[raw['Driver'].isin(top_drivers) & raw['LapTime_s'].between(60, 120)]

# Create the boxplot
plt.figure(figsize=(14, 7))
sns.boxplot(
    data=driver_laps,
    x='Driver',
    y='LapTime_s',
    order=top_drivers,
    palette='Blues_r'
)

plt.title('Lap Time Distribution by Driver (Top 12 Drivers, Active Laps: 60s - 120s)', fontsize=14)
plt.xlabel('Driver', fontsize=12)
plt.ylabel('Lap Time (seconds)', fontsize=12)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Reading the next four cells.** They compare raw lap times, which mix different circuits (a lap of Bahrain
is longer than a lap of Jeddah) and different fuel loads (cars get faster as fuel burns off). That is why
lap time appears to *fall* as tyres age. These cells are a first look only; the tyre effect is measured
properly in section 4, after the fuel correction and within each stint.

In [ ]:
# Filter the raw DataFrame for 'SOFT' compound laps
soft_laps = raw[raw['Compound'] == 'SOFT']
display(soft_laps.head())

In [ ]:
import pandas as pd

# Filter for clean, active racing laps on SOFT and MEDIUM compounds
comparison_laps = raw[raw['Compound'].isin(['SOFT', 'MEDIUM']) & raw['LapTime_s'].between(60, 120)]

# Calculate average lap times and counts
summary_stats = comparison_laps.groupby('Compound')['LapTime_s'].agg(['mean', 'median', 'std', 'count']).reset_index()

display(summary_stats)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Filter active racing laps for SOFT and MEDIUM compounds
vis_laps = raw[raw['Compound'].isin(['SOFT', 'MEDIUM']) & raw['LapTime_s'].between(60, 120)]

plt.figure(figsize=(12, 6))

# Create a scatter plot with regression trends for each compound
sns.lmplot(
    data=vis_laps,
    x='TyreLife',
    y='LapTime_s',
    hue='Compound',
    palette={'SOFT': '#d62728', 'MEDIUM': '#e6b800'},
    scatter_kws={'alpha': 0.3, 's': 15},
    line_kws={'linewidth': 2.5},
    height=6,
    aspect=1.8
)

plt.title("Impact of Tyre Life (Age) on Lap Times (SOFT vs. MEDIUM)", fontsize=14, pad=15)
plt.xlabel("Tyre Life (Laps Completed)", fontsize=12)
plt.ylabel("Lap Time (seconds)", fontsize=12)
plt.xlim(0, 35)
plt.grid(True, linestyle='--', alpha=0.5)

plt.show()

In [ ]:
# Filter for active racing laps
clean_laps = raw[raw['LapTime_s'].between(60, 120)]

# Calculate correlation for each compound
correlations = {}
for compound in clean_laps['Compound'].unique():
    comp_data = clean_laps[clean_laps['Compound'] == compound]
    if len(comp_data) > 1:
        corr = comp_data['TyreLife'].corr(comp_data['LapTime_s'])
        correlations[compound] = {
            'Correlation (r)': corr,
            'Laps Count': len(comp_data)
        }

# Convert to DataFrame for visualization
corr_df = pd.DataFrame(correlations).T.sort_values(by='Correlation (r)', ascending=False)
display(corr_df)

In [ ]:
# Filter raw DataFrame to keep only accurate, active racing laps
accurate_racing_laps = raw[raw['IsAccurate'] & raw['LapTime_s'].between(60, 120)]

# Calculate correlation for each compound
accurate_correlations = {}
for compound in accurate_racing_laps['Compound'].unique():
    comp_data_acc = accurate_racing_laps[accurate_racing_laps['Compound'] == compound]
    if len(comp_data_acc) > 1:
        corr_acc = comp_data_acc['TyreLife'].corr(comp_data_acc['LapTime_s'])
        accurate_correlations[compound] = {
            'Accurate Correlation (r)': corr_acc,
            'Accurate Laps Count': len(comp_data_acc)
        }

# Convert to DataFrame for visualization
corr_acc_df = pd.DataFrame(accurate_correlations).T.sort_values(by='Accurate Correlation (r)', ascending=False)
display(corr_acc_df)

### If the download is blocked

The Formula 1 timing service refuses many requests from cloud computers such as Google Colab. When FastF1
returns nothing for a race, the cell above reads the same FastF1 lap table from the **TracingInsights
archive** on GitHub (MIT licence) and prints `(tracinginsights)` next to the race. That is real timing
data from a second route, and it must be named as a source in your report.

If a race is in neither source (usually only the most recent races), download it once on your own computer:

1. Unzip `f1-pit-engine.zip`, open a terminal in that folder and run `pip install -e .`
2. Run `python -m f1pit.ingest --seasons 2022 2023 2024` (it prints OK or SKIP for each race).
3. Upload every file from `data/races/` to Google Drive, into `MyDrive/f1_project/races/`.
4. Come back here and run the download cell again. It reads the saved files and downloads nothing.

For the full pipeline notebook, run `python -m f1pit.ingest` (all seasons) and upload the same files to
`MyDrive/f1_project/pipeline/data/races/`.

**If a whole season cannot be downloaded, the notebook still finishes.** It prints a warning, trains on the
earlier real races that did load and checks the model on the last ones (`SPLIT_MODE = "by_race"`). Every lap
is real, but the dataset is small and the split is inside one period, so treat those numbers as a practice
run and say so if you show them. The planned result needs all three seasons. A missing season must never be
filled with copies of another season: that tests the model on its own training races.


## 4. Clean and label

- **Dry races only.** A race where anyone used intermediate or wet tyres is removed (wet strategy reacts to rain, not tyre wear).
- **Pit label.** A lap is a pit lap (label 1) only for a tyre stop made as a race decision: the driver entered
  the pits, kept racing afterwards, and left on a different set of tyres. Three kinds of pit entry are not
  counted: a car retiring into the pits; a drive through the pit lane with no tyre change (the field routed
  through it behind the safety car, or a penalty); and a pit visit longer than five minutes (a red flag, when
  every car waits in the pit lane, or a long repair).
- **Rule-forced races** (Qatar 2023 here) are flagged and kept out of training.

In [ ]:
df = raw.copy()
df["RaceID"] = df["Year"].astype(str) + "_" + df["Round"].astype(int).astype(str).str.zfill(2)

wet_races = df.loc[df["Compound"].isin(["INTERMEDIATE", "WET"]), "RaceID"].unique()
df = df[~df["RaceID"].isin(wet_races)]
df = df[df["Compound"].isin(["SOFT", "MEDIUM", "HARD"])]
if "FastF1Generated" in df.columns:
    df = df[~df["FastF1Generated"]]
df = df.dropna(subset=["LapNumber", "TyreLife"])
df = df.sort_values(["RaceID", "Driver", "LapNumber"]).reset_index(drop=True)

grp = df.groupby(["RaceID", "Driver"])
has_next_lap = grp["LapNumber"].shift(-1).notna()
entered = df["PitInTime_s"].notna() & has_next_lap
# the same compound carrying on with its tyre age counting up = the car only drove through the pit lane
same_set = (grp["Compound"].shift(-1) == df["Compound"]) & (grp["TyreLife"].shift(-1) == df["TyreLife"] + 1)
# more than 5 minutes between pit entry and exit = race suspended (red flag) or car in the garage
suspended = (grp["PitOutTime_s"].shift(-1) - df["PitInTime_s"]) > 300
df["PitLap"] = (entered & ~same_set & ~suspended).astype(int)
print(f"Pit entries not counted as tyre stops: {int((entered & same_set).sum())} without a tyre change, "
      f"{int((entered & ~same_set & suspended).sum())} while the race was suspended")

df["TotalLaps"] = df.groupby("RaceID")["LapNumber"].transform("max")
df["RaceProgress"] = df["LapNumber"] / df["TotalLaps"]
df["LapsRemaining"] = df["TotalLaps"] - df["LapNumber"]
df["SafetyCar"] = df["TrackStatus"].str.contains("[467]", regex=True).astype(int)  # 4 = SC, 6/7 = VSC
df["RuleForced"] = [any(y == yy and k in str(e) for yy, k in RULE_FORCED)
                    for y, e in zip(df["Year"], df["EventName"])]

print(f"Removed {len(wet_races)} wet/mixed race(s): {list(wet_races)}")
print(f"Rule-forced races flagged: {sorted(df.loc[df['RuleForced'], 'RaceID'].unique())}")
print(f"Clean dataset: {df['RaceID'].nunique()} races, {len(df):,} laps, {df['PitLap'].sum():,} pit laps")

## 5. Fuel-corrected tyre degradation

Cars get faster as fuel burns off, which hides tyre wear. For each **training** race, a regression with one
baseline per driver separates the lap-time trend over the race (fuel burn plus track evolution) from tyre age.
The slope learned for each circuit is then applied unchanged to the 2024 races, so validation laps never help
estimate their own correction.

**Degradation** is each lap's corrected time minus the stint's baseline: the average of the stint's first
three clean laps, counting only laps already driven, so no lap uses a later one.

In [ ]:
df["Split"] = split_labels(df)
print(f"Split: {SPLIT_MODE}. Validation = {VALID_LABEL}")

def clean_lap_mask(d):
    mask = d["IsAccurate"].astype(bool) & (d["SafetyCar"] == 0) & d["LapTime_s"].notna()
    if "Deleted" in d.columns:
        mask &= ~d["Deleted"].astype(bool)
    return mask

def fuel_slope(race):
    """Seconds per lap from fuel burn and track evolution, from green-flag accurate laps."""
    clean = race[clean_lap_mask(race)]
    if len(clean) < 100:
        return np.nan
    X = [clean["LapNumber"].to_numpy(float)]
    for comp in ["SOFT", "MEDIUM", "HARD"]:                       # tyre age, per compound
        X.append(np.where(clean["Compound"] == comp, clean["TyreLife"], 0.0))
    X = np.column_stack(X + [pd.get_dummies(clean["Driver"]).to_numpy(float),
                             pd.get_dummies(clean["Compound"]).to_numpy(float)])
    coef, *_ = np.linalg.lstsq(X, clean["LapTime_s"].to_numpy(float), rcond=None)
    return float(coef[0])

# slopes from TRAINING races only (rule-forced races excluded), one per circuit, plus an overall fallback
train_races = df[(df["Split"] == "train") & ~df["RuleForced"]]
per_race = pd.Series({rid: fuel_slope(r) for rid, r in train_races.groupby("RaceID")}, dtype=float).dropna()
race_event = train_races.drop_duplicates("RaceID").set_index("RaceID")["EventName"]
slope_by_event = per_race.groupby(race_event.reindex(per_race.index)).median()
df["FuelSlope"] = df["EventName"].map(slope_by_event).fillna(per_race.median())
df["LapTimeCorr_s"] = df["LapTime_s"] - df["FuelSlope"] * (df["LapNumber"] - 1)

def causal_baseline(values, n_laps=3):
    """Running mean of the stint's first n clean laps, using only laps up to the current one."""
    out, total, count = np.full(len(values), np.nan), 0.0, 0
    for i, v in enumerate(values.to_numpy(float)):
        if not np.isnan(v) and count < n_laps:
            total, count = total + v, count + 1
        if count:
            out[i] = total / count
    return pd.Series(out, index=values.index)

df = df.sort_values(["RaceID", "Driver", "LapNumber"]).reset_index(drop=True)
df["CleanLap"] = clean_lap_mask(df)
stint_keys = ["RaceID", "Driver", "Stint"]
clean_times = df["LapTimeCorr_s"].where(df["CleanLap"])
df["StintBase"] = clean_times.groupby([df[k] for k in stint_keys]).transform(causal_baseline)
df["Degradation_s"] = df["LapTimeCorr_s"] - df["StintBase"]

print(f"Fuel slopes estimated from {len(per_race)} training races")
print(f"Median fuel + track effect: {per_race.median():.3f} s per lap (negative = cars get faster)")

## 6. What the training data looks like

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
COLORS = {"SOFT": "#d62728", "MEDIUM": "#e6b800", "HARD": "#7f7f7f"}

def save(fig, name):
    fig.savefig(f"{OUT_DIR}/{name}.png", bbox_inches="tight")
    plt.show()

summary = (df.groupby(["Split", "Year"])
             .agg(Races=("RaceID", "nunique"), Drivers=("Driver", "nunique"),
                  Laps=("LapNumber", "size"), PitLaps=("PitLap", "sum"))
             .assign(**{"Pit lap %": lambda t: (100 * t["PitLaps"] / t["Laps"]).round(2)}))
summary.to_csv(f"{OUT_DIR}/dataset_summary.csv")
display(summary)
display(df[["RaceID", "EventName", "Driver", "LapNumber", "Compound", "TyreLife",
            "LapTime_s", "Degradation_s", "SafetyCar", "PitLap"]].head(10))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# a) class imbalance
counts = df["PitLap"].value_counts().sort_index()
axes[0].bar(["Normal lap", "Pit lap"], counts.values, color=["#9ecae1", "#08519c"])
for i, v in enumerate(counts.values):
    axes[0].text(i, v, f"{v:,}", ha="center", va="bottom")
axes[0].set_title(f"Pit laps are only {100 * counts.get(1, 0) / counts.sum():.1f}% of laps")

# b) degradation curve per compound
d = df[df["CleanLap"] & df["Degradation_s"].between(-3, 5) & (df["TyreLife"] <= 40)]
for comp, color in COLORS.items():
    g = d[d["Compound"] == comp].groupby("TyreLife")["Degradation_s"].agg(["median", "size"])
    g = g[g["size"] >= 30]
    axes[1].plot(g.index, g["median"], color=color, lw=2, label=comp.title())
axes[1].axhline(0, color="black", lw=0.6)
axes[1].set(xlabel="Tyre age (laps)", ylabel="Fuel-corrected time lost (s)",
            title="Pace lost as tyres age")
axes[1].legend(frameon=False)

# c) when pit stops happen
p = df[df["PitLap"] == 1]
axes[2].hist([p.loc[p["SafetyCar"] == 0, "RaceProgress"], p.loc[p["SafetyCar"] == 1, "RaceProgress"]],
             bins=20, stacked=True, color=["#08519c", "#fd8d3c"], label=["Green flag", "Safety car / VSC"])
axes[2].set(xlabel="Race progress (0 = start, 1 = finish)", ylabel="Pit stops",
            title="When drivers pit")
axes[2].legend(frameon=False)
save(fig, "training_data_overview")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Local dictionary of F1 tyre compound colors to ensure the cell is fully self-contained
COLORS_LOCAL = {"SOFT": "#d62728", "MEDIUM": "#e6b800", "HARD": "#7f7f7f"}

# Ensure the required columns exist. If they were lost in the kernel state, recalculate them
if 'Degradation_s' not in df.columns:
    print("Degradation columns missing from DataFrame. Recalculating fuel corrections...")
    df["Split"] = split_labels(df)

    def clean_lap_mask(d):
        mask = d["IsAccurate"].astype(bool) & (d["SafetyCar"] == 0) & d["LapTime_s"].notna()
        if "Deleted" in d.columns:
            mask &= ~d["Deleted"].astype(bool)
        return mask

    def fuel_slope(race):
        clean = race[clean_lap_mask(race)]
        if len(clean) < 100:
            return np.nan
        X = [clean["LapNumber"].to_numpy(float)]
        for comp in ["SOFT", "MEDIUM", "HARD"]:
            X.append(np.where(clean["Compound"] == comp, clean["TyreLife"], 0.0))
        X = np.column_stack(X + [pd.get_dummies(clean["Driver"]).to_numpy(float),
                                 pd.get_dummies(clean["Compound"]).to_numpy(float)])
        coef, *_ = np.linalg.lstsq(X, clean["LapTime_s"].to_numpy(float), rcond=None)
        return float(coef[0])

    train_races = df[(df["Split"] == "train") & ~df["RuleForced"]]
    per_race = pd.Series({rid: fuel_slope(r) for rid, r in train_races.groupby("RaceID")}, dtype=float).dropna()
    race_event = train_races.drop_duplicates("RaceID").set_index("RaceID")["EventName"]
    slope_by_event = per_race.groupby(race_event.reindex(per_race.index)).median()
    df["FuelSlope"] = df["EventName"].map(slope_by_event).fillna(per_race.median())
    df["LapTimeCorr_s"] = df["LapTime_s"] - df["FuelSlope"] * (df["LapNumber" ] - 1)

    def causal_baseline(values, n_laps=3):
        out, total, count = np.full(len(values), np.nan), 0.0, 0
        for i, v in enumerate(values.to_numpy(float)):
            if not np.isnan(v) and count < n_laps:
                total, count = total + v, count + 1
            if count:
                out[i] = total / count
        return pd.Series(out, index=values.index)

    df = df.sort_values(["RaceID", "Driver", "LapNumber"]).reset_index(drop=True)
    df["CleanLap"] = clean_lap_mask(df)
    stint_keys = ["RaceID", "Driver", "Stint"]
    clean_times = df["LapTimeCorr_s"].where(df["CleanLap"])
    df["StintBase"] = clean_times.groupby([df[k] for k in stint_keys]).transform(causal_baseline)
    df["Degradation_s"] = df["LapTimeCorr_s"] - df["StintBase"]

# Filter for clean racing laps with valid degradation within bounds
is_clean = (df["IsAccurate"].astype(bool)) & (df["SafetyCar"] == 0) & (df["Degradation_s"].notna())
if "Deleted" in df.columns:
    is_clean &= ~df["Deleted"].astype(bool)

clean_racing_laps = df[is_clean & df["Degradation_s"].between(-3, 5) & (df["TyreLife"] <= 40)]

plt.figure(figsize=(10, 6))

# Plot a smoothed regression line with 95% confidence intervals for each tyre compound
for comp, color in COLORS_LOCAL.items():
    comp_data = clean_racing_laps[clean_racing_laps["Compound"] == comp]
    sns.lineplot(
        data=comp_data,
        x="TyreLife",
        y="Degradation_s",
        color=color,
        label=comp.title(),
        linewidth=2.5,
        errorbar=("ci", 95)
    )

plt.axhline(0, color="black", linestyle="--", alpha=0.5)
plt.title("Fuel-Corrected Pace Loss (Degradation) vs. Tyre Life", fontsize=14, pad=15)
plt.xlabel("Tyre Life (Laps Completed)", fontsize=12)
plt.ylabel("Pace Lost (seconds)", fontsize=12)
plt.xlim(0, 35)
plt.legend(title="Tyre Compound", frameon=True)
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Ensure COLORS dictionary and save function are defined locally if missing from the global namespace
COLORS_LOCAL = globals().get('COLORS', {"SOFT": "#d62728", "MEDIUM": "#e6b800", "HARD": "#7f7f7f"})

def save_local(fig, name):
    if 'save' in globals():
        globals()['save'](fig, name)
    else:
        import os
        os.makedirs(globals().get('OUT_DIR', './'), exist_ok=True)
        fig.savefig(f"{globals().get('OUT_DIR', './')}/{name}.png", bbox_inches="tight")
        plt.show()

stints = (df.groupby(["RaceID", "Driver", "Stint", "Compound"])["TyreLife"].max()
            .reset_index(name="StintLength"))
fig, ax = plt.subplots(figsize=(7, 4))
data = [stints.loc[stints["Compound"] == c, "StintLength"] for c in COLORS_LOCAL]
bp = ax.boxplot(data, patch_artist=True, showfliers=False)
ax.set_xticks(range(1, len(COLORS_LOCAL) + 1), [c.title() for c in COLORS_LOCAL])
for patch, (comp, c) in zip(bp["boxes"], COLORS_LOCAL.items()):
    patch.set_facecolor(c); patch.set_alpha(0.6)
ax.set(ylabel="Tyre age at end of stint (laps)", title="Harder tyres run longer stints")
save_local(fig, "stint_lengths")

In [ ]:
# Calculate average degradation rate (seconds added per lap of tyre life) for dry compounds
is_clean = (df["IsAccurate"].astype(bool)) & (df["SafetyCar"] == 0) & (df["Degradation_s"].notna())
if "Deleted" in df.columns:
    is_clean &= ~df["Deleted"].astype(bool)

clean_racing_laps = df[is_clean & df["Degradation_s"].between(-3, 5) & (df["TyreLife"] > 0)]

# Compute average degradation rate as the slope (Degradation_s / TyreLife)
clean_racing_laps = clean_racing_laps.copy()
clean_racing_laps['Deg_Rate_per_Lap'] = clean_racing_laps['Degradation_s'] / clean_racing_laps['TyreLife']

# Group by Compound and get statistical metrics
deg_summary = clean_racing_laps.groupby('Compound')['Deg_Rate_per_Lap'].agg([
    ('Average Rate (s/lap)', 'mean'),
    ('Median Rate (s/lap)', 'median'),
    ('Laps Count', 'count')
]).reindex(['SOFT', 'MEDIUM', 'HARD'])

display(deg_summary)

## 7. Baseline model: predict pit laps

**No cheating with the future.** On a pit lap, the lap time and end-of-lap position already show the stop
(the car is slow entering the pit lane). So the model only sees what was known **before** the lap ended:
tyre age, compound, race progress, safety car status, and the **previous** lap's pace, position and gaps.

Two models are compared on the 2024 validation season:
- **Rule-based:** pit when tyre age reaches the typical stint length for that compound at that circuit (learned from 2022–2023).
- **XGBoost:** a gradient-boosted tree model on the features below.

The decision threshold is chosen with cross-validation **on the training seasons only**, never on 2024.

In [ ]:
g = df.groupby(["RaceID", "Driver"])
df["PrevDegradation"] = g["Degradation_s"].shift(1)
df["AvgDegradation3"] = g["Degradation_s"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
df["PrevPosition"] = g["Position"].shift(1)
df["PrevTime_s"] = g["Time_s"].shift(1)

# Drop any previous run columns to ensure re-run idempotence
cols_to_clean = ["AheadTime_s", "AheadPitted", "PrevLap", "AheadPos", "GapAhead_s"]
df = df.drop(columns=[col for col in df.columns if any(col.startswith(c) for c in cols_to_clean)], errors="ignore")

# car ahead at the end of the previous lap: its gap, and whether it pitted on that lap
prev = df[["RaceID", "LapNumber", "Position", "Time_s", "PitLap"]].rename(
    columns={"LapNumber": "PrevLap", "Position": "AheadPos", "Time_s": "AheadTime_s", "PitLap": "AheadPitted"}
)

df["PrevLap"] = df["LapNumber"] - 1
df["AheadPos"] = df["PrevPosition"] - 1

df = df.merge(
    prev.dropna(subset=["AheadPos"]).drop_duplicates(["RaceID", "PrevLap", "AheadPos"]),
    on=["RaceID", "PrevLap", "AheadPos"],
    how="left"
)

# Clean up helper keys
df = df.drop(columns=["PrevLap", "AheadPos"], errors="ignore")

df["GapAhead_s"] = (df["PrevTime_s"] - df["AheadTime_s"]).clip(0, 60)
df["AheadPitted"] = df["AheadPitted"].fillna(0)
df = df.sort_values(["RaceID", "Driver", "LapNumber"]).reset_index(drop=True)

for comp in ["SOFT", "MEDIUM", "HARD"]:
    df[f"Is{comp.title()}"] = (df["Compound"] == comp).astype(int)

FEATURES = [
    "TyreLife", "IsSoft", "IsMedium", "IsHard", "Stint", "RaceProgress", "LapsRemaining",
    "SafetyCar", "PrevDegradation", "AvgDegradation3", "PrevPosition", "GapAhead_s",
    "AheadPitted", "TrackTemp", "AirTemp"
]

model_df = df[(df["LapNumber"] > 1) & ~df["RuleForced"]].copy()
train = model_df[model_df["Split"] == "train"]
valid = model_df[model_df["Split"] == "validation"]
print(f"Training: {train['RaceID'].nunique()} races, {len(train):,} laps, {train['PitLap'].sum():,} pit laps")
print(f"Validation: {valid['RaceID'].nunique()} races, {len(valid):,} laps, {valid['PitLap'].sum():,} pit laps")
for name, part in (("training", train), ("validation", valid)):
    if part.empty or part["PitLap"].sum() == 0:
        raise RuntimeError(
            f"There are no usable {name} laps (or no pit laps in them). Go back to the download "
            "cell and check which races loaded, run it again, then run the cells below it in order."
        )

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve,
                             precision_score, recall_score, f1_score, confusion_matrix)

pos = train["PitLap"].sum()
model = XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8,
                      colsample_bytree=0.8, scale_pos_weight=(len(train) - pos) / pos,
                      eval_metric="aucpr", random_state=RANDOM_SEED, n_jobs=-1)

# threshold from out-of-fold predictions on the TRAINING races only
n_folds = min(5, train["RaceID"].nunique())
oof = cross_val_predict(model, train[FEATURES], train["PitLap"],
                        groups=train["RaceID"], cv=GroupKFold(n_splits=n_folds), method="predict_proba")[:, 1]
prec, rec, thr = precision_recall_curve(train["PitLap"], oof)
f1s = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-9, None)
THRESHOLD = float(thr[np.argmax(f1s)])

model.fit(train[FEATURES], train["PitLap"])
valid = valid.assign(Prob=model.predict_proba(valid[FEATURES])[:, 1])
valid["PredXGB"] = (valid["Prob"] >= THRESHOLD).astype(int)

# rule-based baseline: typical stint length per circuit + compound, learned from training seasons
typical = (stints[stints["RaceID"].isin(train["RaceID"].unique())]
           .merge(df[["RaceID", "EventName"]].drop_duplicates(), on="RaceID")
           .groupby(["EventName", "Compound"])["StintLength"].median().rename("TypicalStint"))
fallback = stints[stints["RaceID"].isin(train["RaceID"].unique())].groupby("Compound")["StintLength"].median()
valid = valid.merge(typical, left_on=["EventName", "Compound"], right_index=True, how="left")
valid["TypicalStint"] = valid["TypicalStint"].fillna(valid["Compound"].map(fallback))
valid["PredRule"] = (valid["TyreLife"].round() == valid["TypicalStint"].round()).astype(int)

def within_two_laps(v, pred_col):
    """Share of real pit stops with a predicted stop within +/-2 laps for the same driver."""
    hits = []
    for (_, _), d in v.groupby(["RaceID", "Driver"]):
        predicted = d.loc[d[pred_col] == 1, "LapNumber"].values
        for lap in d.loc[d["PitLap"] == 1, "LapNumber"]:
            hits.append(bool(len(predicted)) and np.abs(predicted - lap).min() <= 2)
    return float(np.mean(hits)) if hits else float("nan")

y = valid["PitLap"]
results = pd.DataFrame({
    "Rule-based": {"PR-AUC": np.nan, "ROC-AUC": np.nan,
                   "Precision": precision_score(y, valid["PredRule"], zero_division=0),
                   "Recall": recall_score(y, valid["PredRule"]), "F1": f1_score(y, valid["PredRule"]),
                   "Pit within ±2 laps": within_two_laps(valid, "PredRule")},
    "XGBoost": {"PR-AUC": average_precision_score(y, valid["Prob"]), "ROC-AUC": roc_auc_score(y, valid["Prob"]),
                "Precision": precision_score(y, valid["PredXGB"], zero_division=0),
                "Recall": recall_score(y, valid["PredXGB"]), "F1": f1_score(y, valid["PredXGB"]),
                "Pit within ±2 laps": within_two_laps(valid, "PredXGB")},
}).round(3)
no_skill = y.mean()
print(f"Validation = {VALID_LABEL}. No-skill PR-AUC (random guessing) = {no_skill:.3f}")
if SPLIT_MODE == "by_race":
    print("PRACTICE RESULT: a season is missing, so the last races were held out instead of a whole season.")
print(f"Decision threshold (chosen on training races) = {THRESHOLD:.3f}")
results.to_csv(f"{OUT_DIR}/baseline_results.csv")
display(results)

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, confusion_matrix

# Ensure y_true directly matches the length of valid
if "PitLap" in valid.columns:
    y_true = valid["PitLap"].values
elif "y_valid" in globals() and len(y_valid) == len(valid):
    y_true = y_valid
elif "y" in globals() and len(y) == len(valid):
    y_true = y
else:
    raise ValueError(f"Cannot find ground-truth column matching valid length ({len(valid)} rows).")

# Ensure predicted probabilities and classes are populated in valid
if "Prob" not in valid.columns:
    if "ProbXGB" in valid.columns:
        valid["Prob"] = valid["ProbXGB"]
    elif "y_prob" in valid.columns:
        valid["Prob"] = valid["y_prob"]
    elif "model" in globals():
        valid["Prob"] = model.predict_proba(valid[FEATURES])[:, 1]
    else:
        raise KeyError("Neither 'Prob' nor 'ProbXGB' found in valid DataFrame, and 'model' is not defined.")

if "PredXGB" not in valid.columns:
    if "Pred" in valid.columns:
        valid["PredXGB"] = valid["Pred"]
    elif "THRESHOLD" in globals():
        valid["PredXGB"] = (valid["Prob"] >= THRESHOLD).astype(int)
    elif "model" in globals():
        valid["PredXGB"] = model.predict(valid[FEATURES])

# Recalculate no_skill baseline matching the current validation set
no_skill = y_true.sum() / len(y_true)

def save_local(fig, name):
    if 'save' in globals():
        globals()['save'](fig, name)
    else:
        os.makedirs(OUT_DIR, exist_ok=True)
        fig.savefig(f"{OUT_DIR}/{name}.png", bbox_inches="tight")
        plt.show()

# Enable constrained_layout to prevent label overlap
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), constrained_layout=True)

p, r, _ = precision_recall_curve(y_true, valid["Prob"])
axes[0].plot(r, p, color="#08519c", lw=2, label=f"XGBoost (PR-AUC {results.loc['PR-AUC', 'XGBoost']:.2f})")
axes[0].axhline(no_skill, color="grey", ls="--", label=f"Random guess ({no_skill:.2f})")
axes[0].set(xlabel="Recall (share of real stops found)", ylabel="Precision (share of alerts that were real)",
            title=f"Precision-recall on {VALID_TITLE}")
axes[0].legend(frameon=False)

cm = confusion_matrix(y_true, valid["PredXGB"])
axes[1].imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    axes[1].text(j, i, f"{v:,}", ha="center", va="center", color="white" if v > cm.max() / 2 else "black")
axes[1].set(xticks=[0, 1], yticks=[0, 1], xticklabels=["No pit", "Pit"], yticklabels=["No pit", "Pit"],
            xlabel="Predicted", ylabel="Actual", title="XGBoost confusion matrix")

imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
axes[2].barh(imp.index, imp.values, color="#6baed6")
axes[2].set(title="What the model relies on", xlabel="Importance (gain share)")

save_local(fig, "baseline_results")

with open(f"{OUT_DIR}/run_info.json", "w") as f:
    json.dump({"train_seasons": TRAIN_SEASONS, "valid_season": VALID_SEASON, "split_mode": SPLIT_MODE, "valid_races": list(VALID_RACES), "quick": QUICK,
               "threshold": THRESHOLD, "features": FEATURES, "failed_races": failed,
               "results": results.to_dict()}, f, indent=2, default=str)
print(f"Saved charts, tables and run_info.json to {OUT_DIR}")

In [ ]:
# Display the baseline performance metrics comparison
import pandas as pd

display(results)

## 8. How to read these results

- **PR-AUC** is the headline number. With so few pit laps, compare it with the *random guess* line, not with 1.0.
- **Pit within ±2 laps** is the most practical number: how often the model flags a stop close to when the team actually stopped.
- **Accuracy is not shown on purpose.** A model that never predicts a stop would score about 97% accuracy and be useless.
- These are **validation** results. Your final, reportable test is on 2025 (and 2026) with the two-stage LSTM, run once at the end.
- If you ran with `QUICK = True`, the numbers come from only a few races; set `QUICK = False` for results you can report.
- If the results cell printed **PRACTICE RESULT**, a season was missing and the last races were held out instead
  of the 2024 season. Do not report those numbers as the 2024 validation.
